# Notebook 08 — Construcción de `gold.cliente_360`

## Objetivo

Construir la tabla analítica `gold.cliente_360`, que constituye la pieza central del proyecto. Esta tabla agrega en una única fila por cliente todas las variables relevantes para los análisis posteriores: clustering, segmentación, geomarketing, modelización de fidelidad y análisis comercial.

## Arquitectura de la tabla

La tabla se compone de cinco bloques de variables:

| Bloque | Fuente principal | Tipo de variables |
|---|---|---|
| **1. Identificación** | silver.dim_cliente + silver.mapeo_paises | Identificador único, datos descriptivos, tipo de mercado, país |
| **2. Geográfico** | silver.mosaic | Grupo MOSAIC, renta media, perfiles comerciales |
| **3. Económico** | silver.fact_lineas_pedido | Facturación, ticket medio, IVA aportado, número de pedidos |
| **4. Comportamiento** | silver.ventas_minoristas | Operaciones, ratio de repetición, mix de canal, devoluciones |
| **5. Temporal** | silver.tiempo + facts | Recencia, frecuencia, estacionalidad, pedidos en eventos comerciales |

## Población objetivo

La tabla incluye los **3.469 clientes** completos de la cartera (2.082 nacionales + 1.387 internacionales). Esta decisión permite que la misma tabla sirva tanto para análisis técnicos (que filtran por tipo_mercado = 'NACIONAL') como para el análisis comercial de la cartera internacional.

## Metodología de construcción

Se construye de forma incremental, bloque por bloque, validando los volúmenes y la coherencia tras cada paso. Esta estrategia, aunque más lenta que una construcción monolítica, garantiza la trazabilidad y facilita la detección temprana de errores.

## 1. Configuración y conexión a DuckDB

Se establece la conexión con la base de datos y se confirma la disponibilidad de todas las tablas Silver de origen.

In [12]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}\n")

# Verificar que todas las tablas Silver están disponibles
print("=" * 60)
print("VERIFICACIÓN DE TABLAS SILVER DISPONIBLES")
print("=" * 60)
tablas_silver = con.execute("""
    SELECT table_name, estimated_size AS filas_aprox
    FROM duckdb_tables
    WHERE schema_name = 'silver'
    ORDER BY table_name
""").fetchdf()
print(tablas_silver.to_string(index=False))

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb

VERIFICACIÓN DE TABLAS SILVER DISPONIBLES
        table_name  filas_aprox
       dim_cliente         3469
fact_lineas_pedido        33353
      mapeo_paises          156
            mosaic         6457
            tiempo         1461
 ventas_minoristas      2870570


In [13]:
print("=" * 60)
print("VOLÚMENES EXACTOS DE LAS TABLAS SILVER")
print("=" * 60)

tablas = ['dim_cliente', 'fact_lineas_pedido', 'ventas_minoristas', 'tiempo', 'mosaic', 'mapeo_paises']

for t in tablas:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM silver.{t}").fetchone()[0]
        print(f"  silver.{t:.<25s} {n:>12,} filas")
    except Exception as e:
        print(f"  silver.{t:.<25s} ❌ ERROR: {e}")

VOLÚMENES EXACTOS DE LAS TABLAS SILVER
  silver.dim_cliente..............        3,469 filas
  silver.fact_lineas_pedido.......       33,353 filas
  silver.ventas_minoristas........    2,870,570 filas
  silver.tiempo...................        1,461 filas
  silver.mosaic...................        6,457 filas
  silver.mapeo_paises.............          156 filas


## 2. Bloque 1 — Identificación

Se construye la versión inicial de `gold.cliente_360` con las variables de identificación del cliente: identificador único, datos descriptivos, clasificación de mercado (NACIONAL / INTERNACIONAL) y país. El país se asigna mediante una lógica condicional: 'España' para los clientes nacionales (independientemente del código de provincia), y resultado del cruce con `silver.mapeo_paises` para los internacionales. Los casos sin match (clientes internacionales con código de provincia no documentado) se etiquetan como 'Sin identificar' para garantizar que el campo siempre tenga valor.

In [14]:
print("Construyendo gold.cliente_360 — Bloque 1: Identificación\n")

con.execute("""
    CREATE OR REPLACE TABLE gold.cliente_360 AS
    SELECT
        -- Identificador único
        d.id_cliente,
        
        -- Datos descriptivos del cliente
        d.nombre_cliente,
        d.nombre_comercial_cliente,
        d.localidad_cliente,
        d.codigo_postal_norm,
        d.codigo_provincia_cliente,
        
        -- Clasificación de mercado y flag
        d.tipo_mercado,
        d.es_cliente_espanol,
        
        -- País: España para nacionales; lookup para internacionales
        CASE
            WHEN d.tipo_mercado = 'NACIONAL' THEN 'España'
            WHEN mp.pais IS NOT NULL THEN mp.pais
            ELSE 'Sin identificar'
        END AS pais
        
    FROM silver.dim_cliente d
    LEFT JOIN silver.mapeo_paises mp ON d.codigo_provincia_cliente = mp.codigo
""")

print("✅ gold.cliente_360 creada con el Bloque 1 (Identificación)")
n = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]
print(f"   Filas: {n:,}")

Construyendo gold.cliente_360 — Bloque 1: Identificación

✅ gold.cliente_360 creada con el Bloque 1 (Identificación)
   Filas: 3,469


## 3. Validación del Bloque 1

In [15]:
print("=" * 60)
print("VALIDACIÓN 1 — Volúmenes")
print("=" * 60)

resultado = con.execute("""
    SELECT 
        COUNT(*) AS total_clientes,
        COUNT(DISTINCT id_cliente) AS ids_unicos,
        SUM(CASE WHEN tipo_mercado = 'NACIONAL'      THEN 1 ELSE 0 END) AS nacionales,
        SUM(CASE WHEN tipo_mercado = 'INTERNACIONAL' THEN 1 ELSE 0 END) AS internacionales
    FROM gold.cliente_360
""").fetchdf()
print(resultado.T.to_string(header=False))

print("\n  ✓ Esperado: total = 3.469 = ids_unicos = 2.082 + 1.387")

VALIDACIÓN 1 — Volúmenes
total_clientes   3469.0
ids_unicos       3469.0
nacionales       2082.0
internacionales  1387.0

  ✓ Esperado: total = 3.469 = ids_unicos = 2.082 + 1.387


In [16]:
print("=" * 60)
print("VALIDACIÓN 2 — Distribución por país")
print("=" * 60)

por_pais = con.execute("""
    SELECT 
        pais,
        tipo_mercado,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    GROUP BY pais, tipo_mercado
    ORDER BY num_clientes DESC
""").fetchdf()
print(por_pais.to_string(index=False))

print("\n  ✓ Esperado:")
print("    · España + NACIONAL = 2.082")
print("    · Resto de países + INTERNACIONAL = 1.387 en total")
print("    · 'Sin identificar' + INTERNACIONAL ≈ 31")

VALIDACIÓN 2 — Distribución por país
                  pais  tipo_mercado  num_clientes
                España      NACIONAL          2082
                Italia INTERNACIONAL           629
              Portugal INTERNACIONAL           335
               Polonia INTERNACIONAL            83
               Bélgica INTERNACIONAL            62
               Noruega INTERNACIONAL            56
                Canadá INTERNACIONAL            55
          Países Bajos INTERNACIONAL            46
       Sin identificar INTERNACIONAL            31
  España (error carga) INTERNACIONAL            10
                 Rusia INTERNACIONAL             7
           Reino Unido INTERNACIONAL             6
                México INTERNACIONAL             4
              Alemania INTERNACIONAL             4
               Francia INTERNACIONAL             3
               Austria INTERNACIONAL             2
               Andorra INTERNACIONAL             2
                Israel INTERNACIONAL         

In [17]:
print("=" * 60)
print("ESQUEMA ACTUAL DE gold.cliente_360")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))
print(f"\nTotal columnas: {len(esquema)}")

ESQUEMA ACTUAL DE gold.cliente_360
             column_name data_type
              id_cliente   VARCHAR
          nombre_cliente   VARCHAR
nombre_comercial_cliente   VARCHAR
       localidad_cliente   VARCHAR
      codigo_postal_norm   VARCHAR
codigo_provincia_cliente   VARCHAR
            tipo_mercado   VARCHAR
      es_cliente_espanol   BOOLEAN
                    pais   VARCHAR

Total columnas: 9


## 4. Bloque 2 — Geográfico

Se enriquece la tabla con las variables sociodemográficas de MOSAIC para los clientes con código postal cubierto por Experian. La operación se realiza mediante un LEFT JOIN para preservar el universo completo de 3.469 clientes: los que no cruzan con MOSAIC (clientes internacionales y 58 nacionales con CPs no cubiertos) quedan con valores NULL en estas columnas y se identifican con un flag `tiene_mosaic = FALSE` para facilitar el filtrado en análisis posteriores.

In [18]:
print("Reconstruyendo Bloque 2 (Geográfico) con JOIN restringido a NACIONAL...\n")

con.execute("""
    CREATE OR REPLACE TABLE gold.cliente_360 AS
    SELECT
        c.id_cliente,
        c.nombre_cliente,
        c.nombre_comercial_cliente,
        c.localidad_cliente,
        c.codigo_postal_norm,
        c.codigo_provincia_cliente,
        c.tipo_mercado,
        c.es_cliente_espanol,
        c.pais,
        
        -- Bloque 2: MOSAIC solo aplica a NACIONALES
        m.mosaic_grupo,
        m.mosaic_grupo_peso,
        m.mosaic_segmento,
        m.mosaic_segmento_peso,
        m.renta_media,
        COALESCE(m.perfil_premium,         FALSE) AS perfil_premium,
        COALESCE(m.perfil_familiar_joven,  FALSE) AS perfil_familiar_joven,
        COALESCE(m.perfil_turistico,       FALSE) AS perfil_turistico,
        COALESCE(m.perfil_rural,           FALSE) AS perfil_rural,
        COALESCE(m.perfil_precio_sensible, FALSE) AS perfil_precio_sensible,
        
        -- Flag: tiene MOSAIC SOLO si es nacional y el JOIN encontró match
        CASE 
            WHEN c.tipo_mercado = 'NACIONAL' AND m.codigo_postal_norm IS NOT NULL 
            THEN TRUE 
            ELSE FALSE 
        END AS tiene_mosaic
        
    FROM gold.cliente_360 c
    LEFT JOIN silver.mosaic m 
        ON c.codigo_postal_norm = m.codigo_postal_norm
        AND c.tipo_mercado = 'NACIONAL'
""")

print("✅ Bloque 2 corregido — MOSAIC solo aplica a cartera nacional")

Reconstruyendo Bloque 2 (Geográfico) con JOIN restringido a NACIONAL...

✅ Bloque 2 corregido — MOSAIC solo aplica a cartera nacional


## 5. Validación del Bloque 2

In [19]:
print("=" * 60)
print("VALIDACIÓN 3 — Cobertura MOSAIC tras el JOIN")
print("=" * 60)

cobertura = con.execute("""
    SELECT 
        tipo_mercado,
        tiene_mosaic,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    GROUP BY tipo_mercado, tiene_mosaic
    ORDER BY tipo_mercado, tiene_mosaic DESC
""").fetchdf()
print(cobertura.to_string(index=False))

print("\n  ✓ Esperado:")
print("    · NACIONAL + tiene_mosaic=TRUE  ≈ 2.024")
print("    · NACIONAL + tiene_mosaic=FALSE ≈ 58")
print("    · INTERNACIONAL + tiene_mosaic=FALSE = 1.387 (MOSAIC solo cubre España)")

VALIDACIÓN 3 — Cobertura MOSAIC tras el JOIN
 tipo_mercado  tiene_mosaic  num_clientes
INTERNACIONAL         False          1387
     NACIONAL          True          2024
     NACIONAL         False            58

  ✓ Esperado:
    · NACIONAL + tiene_mosaic=TRUE  ≈ 2.024
    · NACIONAL + tiene_mosaic=FALSE ≈ 58
    · INTERNACIONAL + tiene_mosaic=FALSE = 1.387 (MOSAIC solo cubre España)


In [20]:
print("=" * 60)
print("VALIDACIÓN 4 — Distribución de la cartera nacional por grupo MOSAIC")
print("=" * 60)

por_grupo = con.execute("""
    SELECT 
        COALESCE(mosaic_grupo, 'SIN MOSAIC') AS grupo,
        COUNT(*) AS num_clientes,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
        ROUND(AVG(renta_media), 0) AS renta_media_avg
    FROM gold.cliente_360
    WHERE tipo_mercado = 'NACIONAL'
    GROUP BY grupo
    ORDER BY num_clientes DESC
""").fetchdf()
print(por_grupo.to_string(index=False))

print("\n  ✓ Esperado: distribución similar a la población general")
print("    pero con sesgo hacia zonas urbanas con presencia comercial.")

VALIDACIÓN 4 — Distribución de la cartera nacional por grupo MOSAIC
     grupo  num_clientes   pct  renta_media_avg
         H           503 24.16          23787.0
         C           313 15.03          27695.0
         B           229 11.00          27190.0
         I           203  9.75          20854.0
         A           181  8.69          31908.0
         G           178  8.55          27055.0
         E           134  6.44          28434.0
         D           113  5.43          27757.0
         J            83  3.99          21473.0
SIN MOSAIC            58  2.79              NaN
         F            44  2.11          24971.0
         K            41  1.97          20555.0
         U             2  0.10          24810.0

  ✓ Esperado: distribución similar a la población general
    pero con sesgo hacia zonas urbanas con presencia comercial.


In [21]:
print("=" * 60)
print("VALIDACIÓN 5 — Flags de perfil comercial en la cartera nacional")
print("=" * 60)

flags = con.execute("""
    SELECT 
        SUM(CASE WHEN perfil_premium         THEN 1 ELSE 0 END) AS premium,
        SUM(CASE WHEN perfil_familiar_joven  THEN 1 ELSE 0 END) AS familiar_joven,
        SUM(CASE WHEN perfil_turistico       THEN 1 ELSE 0 END) AS turistico,
        SUM(CASE WHEN perfil_rural           THEN 1 ELSE 0 END) AS rural,
        SUM(CASE WHEN perfil_precio_sensible THEN 1 ELSE 0 END) AS precio_sensible
    FROM gold.cliente_360
    WHERE tipo_mercado = 'NACIONAL'
""").fetchdf()
print(flags.T.to_string(header=False))

print("\n  ✓ Estos números (sobre la cartera nacional) serán la base del")
print("    análisis de segmentación geomarketing posterior.")

VALIDACIÓN 5 — Flags de perfil comercial en la cartera nacional
premium          494.0
familiar_joven   247.0
turistico         44.0
rural            124.0
precio_sensible  747.0

  ✓ Estos números (sobre la cartera nacional) serán la base del
    análisis de segmentación geomarketing posterior.


In [22]:
print("=" * 60)
print("ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 2)")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))
print(f"\nTotal columnas: {len(esquema)}")

ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 2)
             column_name data_type
              id_cliente   VARCHAR
          nombre_cliente   VARCHAR
nombre_comercial_cliente   VARCHAR
       localidad_cliente   VARCHAR
      codigo_postal_norm   VARCHAR
codigo_provincia_cliente   VARCHAR
            tipo_mercado   VARCHAR
      es_cliente_espanol   BOOLEAN
                    pais   VARCHAR
            mosaic_grupo   VARCHAR
       mosaic_grupo_peso    DOUBLE
         mosaic_segmento   VARCHAR
    mosaic_segmento_peso    DOUBLE
             renta_media    DOUBLE
          perfil_premium   BOOLEAN
   perfil_familiar_joven   BOOLEAN
        perfil_turistico   BOOLEAN
            perfil_rural   BOOLEAN
  perfil_precio_sensible   BOOLEAN
            tiene_mosaic   BOOLEAN

Total columnas: 20


In [23]:
print("=" * 70)
print("INVESTIGACIÓN — 58 clientes nacionales sin MOSAIC")
print("=" * 70)

sin_mosaic = con.execute("""
    SELECT 
        codigo_postal_norm,
        SUBSTR(codigo_postal_norm, 1, 2) AS provincia_cp,
        COUNT(*) AS num_clientes,
        STRING_AGG(DISTINCT localidad_cliente, ' | ') AS localidades
    FROM gold.cliente_360
    WHERE tipo_mercado = 'NACIONAL' AND tiene_mosaic = FALSE
    GROUP BY codigo_postal_norm
    ORDER BY codigo_postal_norm
""").fetchdf()

print(f"CPs distintos sin cobertura: {len(sin_mosaic)}")
print(f"Total clientes afectados: {sin_mosaic['num_clientes'].sum()}\n")
print(sin_mosaic.to_string(index=False))

print("\n" + "=" * 70)
print("DISTRIBUCIÓN POR PROVINCIA")
print("=" * 70)
por_prov = con.execute("""
    SELECT 
        SUBSTR(codigo_postal_norm, 1, 2) AS provincia_cp,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    WHERE tipo_mercado = 'NACIONAL' AND tiene_mosaic = FALSE
    GROUP BY provincia_cp
    ORDER BY num_clientes DESC
""").fetchdf()
print(por_prov.to_string(index=False))

INVESTIGACIÓN — 58 clientes nacionales sin MOSAIC
CPs distintos sin cobertura: 39
Total clientes afectados: 58

codigo_postal_norm provincia_cp  num_clientes                       localidades
             03181           03             1                        TORREVIEJA
             04149           04             1                       AGUA AMARGA
             07701           07             2                       MAHON | MAO
             07750           07             1                         FERRERIES
             08242           08             1                           MANRESA
             08401           08             2                        GRANOLLERS
             10440           10             1             ALDEANUEVA DE LA VERA
             10460           10             1             JARANDILLA DE LA VERA
             11139           11             2           CHICLANA DE LA FRONTERA
             15259           15             1                      SERRES MUROS
        

## 6. Bloque 3 — Económico

Se enriquece la tabla con las variables económicas derivadas de `silver.fact_lineas_pedido`. La metodología aplicada respeta la naturaleza de los campos de cabecera (importes con sufijo `_pedido` que se repiten en cada línea del pedido) documentada por Selmark: el cálculo se realiza en dos pasos sucesivos para evitar la inflación artificial de los importes.

**Paso 1 (CTE pedidos_unicos)**: se agrupa por `id_pedido` aplicando `MAX()` a los campos de cabecera, obteniendo un único registro por pedido con el importe real facturado.

**Paso 2 (CTE metricas_cliente)**: se agrupa por `id_cliente` sumando los pedidos únicos para calcular la facturación total, número de pedidos y ticket medio.

Se utiliza `LEFT JOIN` para preservar el universo completo de 3.469 clientes: los que no tienen actividad en `fact_lineas_pedido` quedan con NULL en las métricas económicas y se identifican con el flag `tiene_actividad_economica = FALSE`.

In [68]:
print("Reconstruyendo Bloque 3 limpio desde Bloques 1+2...\n")

con.execute("""
    CREATE OR REPLACE TABLE gold.cliente_360 AS
    WITH base AS (
        -- Recuperamos SOLO los Bloques 1 y 2 (las primeras 20 columnas válidas)
        SELECT
            id_cliente,
            nombre_cliente,
            nombre_comercial_cliente,
            localidad_cliente,
            codigo_postal_norm,
            codigo_provincia_cliente,
            tipo_mercado,
            es_cliente_espanol,
            pais,
            mosaic_grupo,
            mosaic_grupo_peso,
            mosaic_segmento,
            mosaic_segmento_peso,
            renta_media,
            perfil_premium,
            perfil_familiar_joven,
            perfil_turistico,
            perfil_rural,
            perfil_precio_sensible,
            tiene_mosaic
        FROM gold.cliente_360
    ),
    pedidos_unicos AS (
        SELECT
            id_cliente,
            id_pedido,
            MAX(importe_total_pedido)          AS importe_total_pedido,
            MAX(importe_bruto_pedido)          AS importe_bruto_pedido,
            MAX(importe_base_imponible_pedido) AS importe_base_imp_pedido,
            MAX(importe_iva_pedido)            AS importe_iva_pedido
        FROM silver.fact_lineas_pedido
        WHERE precio_unidad > 0
        GROUP BY id_cliente, id_pedido
    ),
    metricas_b2b AS (
        SELECT
            id_cliente,
            COUNT(DISTINCT id_pedido)            AS num_pedidos_b2b,
            SUM(importe_total_pedido)            AS facturacion_b2b,
            ROUND(AVG(importe_total_pedido), 2)  AS ticket_medio_b2b,
            SUM(importe_iva_pedido)              AS iva_aportado_b2b,
            SUM(importe_bruto_pedido)            AS importe_bruto_total_b2b,
            SUM(importe_base_imp_pedido)         AS base_imponible_total_b2b
        FROM pedidos_unicos
        GROUP BY id_cliente
    )
    SELECT
        b.*,
        COALESCE(mb.num_pedidos_b2b, 0)          AS num_pedidos_b2b,
        mb.facturacion_b2b,
        mb.ticket_medio_b2b,
        mb.iva_aportado_b2b,
        mb.importe_bruto_total_b2b,
        mb.base_imponible_total_b2b,
        CASE WHEN mb.id_cliente IS NOT NULL THEN TRUE ELSE FALSE END AS tiene_facturacion_b2b
    FROM base b
    LEFT JOIN metricas_b2b mb ON b.id_cliente = mb.id_cliente
""")

print("✅ gold.cliente_360 reconstruida desde cero")
n = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]
n_cols = con.execute("""
    SELECT COUNT(*) FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
""").fetchone()[0]
print(f"   Filas: {n:,}")
print(f"   Columnas: {n_cols}  (esperado: 27)")

Reconstruyendo Bloque 3 limpio desde Bloques 1+2...

✅ gold.cliente_360 reconstruida desde cero
   Filas: 3,469
   Columnas: 27  (esperado: 27)


## 7. Validación del Bloque 3

In [74]:
print("=" * 60)
print("VALIDACIÓN 6 — Cobertura de facturación B2B")
print("=" * 60)

cobertura = con.execute("""
    SELECT 
        tipo_mercado,
        tiene_facturacion_b2b,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    GROUP BY tipo_mercado, tiene_facturacion_b2b
    ORDER BY tipo_mercado, tiene_facturacion_b2b DESC
""").fetchdf()
print(cobertura.to_string(index=False))

print("\n  ✓ Esperado:")
print("    · NACIONAL + tiene_facturacion_b2b=TRUE  ≈ ~80 clientes (B2B españoles grandes)")
print("    · INTERNACIONAL + tiene_facturacion_b2b=TRUE ≈ ~150 clientes (distribuidores)")
print("    · Total con facturación B2B: 234 clientes")

VALIDACIÓN 6 — Cobertura de facturación B2B
 tipo_mercado  tiene_facturacion_b2b  num_clientes
INTERNACIONAL                   True           118
INTERNACIONAL                  False          1269
     NACIONAL                   True           116
     NACIONAL                  False          1966

  ✓ Esperado:
    · NACIONAL + tiene_facturacion_b2b=TRUE  ≈ ~80 clientes (B2B españoles grandes)
    · INTERNACIONAL + tiene_facturacion_b2b=TRUE ≈ ~150 clientes (distribuidores)
    · Total con facturación B2B: 234 clientes


In [75]:
print("=" * 60)
print("VALIDACIÓN 7 — Top 10 clientes por facturación B2B")
print("=" * 60)

top10 = con.execute("""
    SELECT 
        id_cliente,
        nombre_cliente,
        pais,
        num_pedidos_b2b,
        ROUND(facturacion_b2b, 2) AS facturacion_eur,
        ROUND(ticket_medio_b2b, 2) AS ticket_medio
    FROM gold.cliente_360
    WHERE tiene_facturacion_b2b = TRUE
    ORDER BY facturacion_b2b DESC
    LIMIT 10
""").fetchdf()
print(top10.to_string(index=False))

VALIDACIÓN 7 — Top 10 clientes por facturación B2B
id_cliente                                 nombre_cliente            pais  num_pedidos_b2b  facturacion_eur  ticket_medio
      5728                          EL CORTE INGLES, S.A.          España             1102       2285893.09       2074.31
      1310             SUTTON CHEMICALS & TEXTILES UL LTD     Reino Unido               15       1259171.69      83944.78
     31249               IP LUKIYANENKOV IGOR NIKOLAEVICK           Rusia               81        905944.29      11184.50
      7571                      BELLISIMA BEACHWEAR, S.L.          España               50        757323.03      15146.46
     31248                       IP ZAVALIN ALEXEY ILYICH           Rusia               31        393949.70      12708.05
      1174                             SAS RESERVOIR MODE         Francia             1373        318656.20        232.09
      1241                                 DONET SPOL SRO República Checa              316     

In [76]:
print("=" * 60)
print("VALIDACIÓN 8 — Estadísticas globales de las variables económicas B2B")
print("=" * 60)

stats = con.execute("""
    SELECT 
        'facturacion_b2b' AS metrica,
        ROUND(MIN(facturacion_b2b), 2)    AS minimo,
        ROUND(AVG(facturacion_b2b), 2)    AS media,
        ROUND(MEDIAN(facturacion_b2b), 2) AS mediana,
        ROUND(MAX(facturacion_b2b), 2)    AS maximo,
        ROUND(SUM(facturacion_b2b), 2)    AS total
    FROM gold.cliente_360 WHERE tiene_facturacion_b2b = TRUE
    UNION ALL
    SELECT 
        'ticket_medio_b2b',
        ROUND(MIN(ticket_medio_b2b), 2),
        ROUND(AVG(ticket_medio_b2b), 2),
        ROUND(MEDIAN(ticket_medio_b2b), 2),
        ROUND(MAX(ticket_medio_b2b), 2),
        NULL
    FROM gold.cliente_360 WHERE tiene_facturacion_b2b = TRUE
    UNION ALL
    SELECT 
        'num_pedidos_b2b',
        MIN(num_pedidos_b2b),
        ROUND(AVG(num_pedidos_b2b), 2),
        MEDIAN(num_pedidos_b2b),
        MAX(num_pedidos_b2b),
        SUM(num_pedidos_b2b)
    FROM gold.cliente_360 WHERE tiene_facturacion_b2b = TRUE
""").fetchdf()
print(stats.to_string(index=False))

print("\n  ✓ Esperado:")
print("    · SUM num_pedidos_b2b ≈ 4.104 (los pedidos B2B con precio>0)")
print("    · MIN facturacion_b2b negativo (2 clientes con devoluciones netas)")
print("    · MAX facturacion_b2b ~2,3M € (El Corte Inglés)")

VALIDACIÓN 8 — Estadísticas globales de las variables económicas B2B
         metrica    minimo    media  mediana     maximo      total
 facturacion_b2b -19178.75 33203.62   826.42 2285893.09 7769646.48
ticket_medio_b2b  -9589.38  1945.32   590.87   83944.78        NaN
 num_pedidos_b2b      1.00    17.54     1.00    1373.00    4104.00

  ✓ Esperado:
    · SUM num_pedidos_b2b ≈ 4.104 (los pedidos B2B con precio>0)
    · MIN facturacion_b2b negativo (2 clientes con devoluciones netas)
    · MAX facturacion_b2b ~2,3M € (El Corte Inglés)


In [77]:
print("=" * 60)
print("ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 3)")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))
print(f"\nTotal columnas: {len(esquema)}")

ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 3)
             column_name data_type
              id_cliente   VARCHAR
          nombre_cliente   VARCHAR
nombre_comercial_cliente   VARCHAR
       localidad_cliente   VARCHAR
      codigo_postal_norm   VARCHAR
codigo_provincia_cliente   VARCHAR
            tipo_mercado   VARCHAR
      es_cliente_espanol   BOOLEAN
                    pais   VARCHAR
            mosaic_grupo   VARCHAR
       mosaic_grupo_peso    DOUBLE
         mosaic_segmento   VARCHAR
    mosaic_segmento_peso    DOUBLE
             renta_media    DOUBLE
          perfil_premium   BOOLEAN
   perfil_familiar_joven   BOOLEAN
        perfil_turistico   BOOLEAN
            perfil_rural   BOOLEAN
  perfil_precio_sensible   BOOLEAN
            tiene_mosaic   BOOLEAN
         num_pedidos_b2b    BIGINT
         facturacion_b2b    DOUBLE
        ticket_medio_b2b    DOUBLE
        iva_aportado_b2b    DOUBLE
 importe_bruto_total_b2b    DOUBLE
base_imponible_total_b2b    DOUBLE
   t

## 8. Bloque 4 — Comportamiento

Se enriquece la tabla con variables de comportamiento operativo derivadas de `silver.ventas_minoristas`. Este bloque recupera 3.054 clientes con actividad operativa real, que es la población más amplia con la que trabajará el clustering posterior.

Las variables se construyen separando las operaciones de venta de las devoluciones (identificadas por `es_devolucion = TRUE` en silver.ventas_minoristas). Las cantidades de devolución se convierten a valores positivos para facilitar la interpretación. El ratio de repetición, calculado como porcentaje de operaciones de tipo 'Repetición' sobre el total, será una variable clave en el análisis de fidelidad.

Se identifican adicionalmente las cuentas corporativas digitales (Ecommerce, Depósito y ECI) con un flag específico, dado que no son comparables con clientes B2B convencionales por agregar ventas a consumidor final.

In [79]:
print("Añadiendo Bloque 4 (Comportamiento) a gold.cliente_360...\n")
print("⚠ Esto puede tardar 15-25 segundos (procesa 2,87 M filas)\n")

con.execute("""
    CREATE OR REPLACE TABLE gold.cliente_360 AS
    WITH base AS (
        SELECT * FROM gold.cliente_360
    ),
    metricas_comportamiento AS (
        SELECT
            id_cliente,
            
            -- Volumen de actividad (sin devoluciones)
            SUM(CASE WHEN es_devolucion = FALSE THEN 1 ELSE 0 END)             AS num_operaciones,
            SUM(CASE WHEN es_devolucion = FALSE THEN cantidad_neta ELSE 0 END) AS unidades_vendidas,
            
            -- Devoluciones (en positivo para legibilidad)
            SUM(CASE WHEN es_devolucion = TRUE  THEN 1 ELSE 0 END)              AS num_operaciones_devol,
            ABS(SUM(CASE WHEN es_devolucion = TRUE  THEN cantidad_neta ELSE 0 END)) AS unidades_devueltas,
            
            -- Variedad de productos
            COUNT(DISTINCT CASE WHEN es_devolucion = FALSE THEN id_sku END)      AS num_skus_distintos,
            COUNT(DISTINCT CASE WHEN es_devolucion = FALSE THEN cod_modelo END)  AS num_modelos_distintos,
            COUNT(DISTINCT id_temporada)                                          AS num_temporadas_activas,
            
            -- Mix de canal: porcentaje de operaciones por tipo de pedido
            ROUND(100.0 * SUM(CASE WHEN id_tipo_pedido = '1'   THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_pedidos_temporada,
            ROUND(100.0 * SUM(CASE WHEN id_tipo_pedido = '2'   THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_pedidos_repeticion,
            
            -- Canal principal: el tipo de pedido más frecuente
            MODE(desc_tipo_pedido) AS canal_principal,
            
            -- Flag: ¿el cliente opera en canales digitales corporativos?
            MAX(CASE WHEN UPPER(desc_tipo_pedido) IN ('ECOMMERCE', 'DEPÓSITO', 'ECI') THEN 1 ELSE 0 END)::BOOLEAN AS es_cuenta_corporativa
            
        FROM silver.ventas_minoristas
        GROUP BY id_cliente
    )
    SELECT
        b.*,
        COALESCE(mc.num_operaciones, 0)               AS num_operaciones,
        COALESCE(mc.unidades_vendidas, 0)             AS unidades_vendidas,
        COALESCE(mc.num_operaciones_devol, 0)         AS num_operaciones_devol,
        COALESCE(mc.unidades_devueltas, 0)            AS unidades_devueltas,
        
        -- Ratio devoluciones (sobre unidades totales movidas)
        CASE 
            WHEN COALESCE(mc.unidades_vendidas, 0) + COALESCE(mc.unidades_devueltas, 0) = 0 THEN NULL
            ELSE ROUND(
                100.0 * COALESCE(mc.unidades_devueltas, 0) /
                (COALESCE(mc.unidades_vendidas, 0) + COALESCE(mc.unidades_devueltas, 0)),
                2
            )
        END AS pct_ratio_devoluciones,
        
        COALESCE(mc.num_skus_distintos, 0)            AS num_skus_distintos,
        COALESCE(mc.num_modelos_distintos, 0)         AS num_modelos_distintos,
        COALESCE(mc.num_temporadas_activas, 0)        AS num_temporadas_activas,
        mc.pct_pedidos_temporada,
        mc.pct_pedidos_repeticion,
        mc.canal_principal,
        COALESCE(mc.es_cuenta_corporativa, FALSE)     AS es_cuenta_corporativa,
        
        -- Flag: tiene actividad operativa en ventas_minoristas
        CASE WHEN mc.id_cliente IS NOT NULL THEN TRUE ELSE FALSE END AS tiene_actividad_minorista
        
    FROM base b
    LEFT JOIN metricas_comportamiento mc ON b.id_cliente = mc.id_cliente
""")

print("✅ Bloque 4 añadido")
n = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]
n_cols = con.execute("""
    SELECT COUNT(*) FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
""").fetchone()[0]
print(f"   Filas: {n:,}")
print(f"   Columnas: {n_cols}  (esperado: 39)")

Añadiendo Bloque 4 (Comportamiento) a gold.cliente_360...

⚠ Esto puede tardar 15-25 segundos (procesa 2,87 M filas)

✅ Bloque 4 añadido
   Filas: 3,469
   Columnas: 40  (esperado: 39)


## 9. Validación del Bloque 4

In [80]:
print("=" * 60)
print("VALIDACIÓN 9 — Cobertura de actividad minorista")
print("=" * 60)

cobertura = con.execute("""
    SELECT 
        tipo_mercado,
        tiene_actividad_minorista,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    GROUP BY tipo_mercado, tiene_actividad_minorista
    ORDER BY tipo_mercado, tiene_actividad_minorista DESC
""").fetchdf()
print(cobertura.to_string(index=False))

print("\n  ✓ Esperado: ~3.054 clientes con actividad minorista (TRUE)")

VALIDACIÓN 9 — Cobertura de actividad minorista
 tipo_mercado  tiene_actividad_minorista  num_clientes
INTERNACIONAL                       True          1176
INTERNACIONAL                      False           211
     NACIONAL                       True          1875
     NACIONAL                      False           207

  ✓ Esperado: ~3.054 clientes con actividad minorista (TRUE)


In [81]:
print("=" * 60)
print("VALIDACIÓN 10 — Estadísticas globales del bloque comportamiento")
print("=" * 60)

stats = con.execute("""
    SELECT 
        ROUND(AVG(num_operaciones), 0)         AS media_operaciones,
        ROUND(AVG(unidades_vendidas), 0)       AS media_unidades_vendidas,
        ROUND(AVG(unidades_devueltas), 0)      AS media_unidades_devueltas,
        ROUND(AVG(pct_ratio_devoluciones), 2)  AS media_ratio_devol,
        ROUND(AVG(num_skus_distintos), 0)      AS media_skus,
        ROUND(AVG(pct_pedidos_temporada), 2)   AS media_pct_temporada,
        ROUND(AVG(pct_pedidos_repeticion), 2)  AS media_pct_repeticion,
        SUM(CASE WHEN es_cuenta_corporativa THEN 1 ELSE 0 END) AS cuentas_corporativas
    FROM gold.cliente_360
    WHERE tiene_actividad_minorista = TRUE
""").fetchdf()
print(stats.T.to_string(header=False))

VALIDACIÓN 10 — Estadísticas globales del bloque comportamiento
media_operaciones          889.00
media_unidades_vendidas   1609.00
media_unidades_devueltas    67.00
media_ratio_devol            3.88
media_skus                 684.00
media_pct_temporada         59.59
media_pct_repeticion        31.17
cuentas_corporativas        15.00


In [82]:
print("=" * 60)
print("VALIDACIÓN 11 — Distribución por canal principal")
print("=" * 60)

canales = con.execute("""
    SELECT 
        canal_principal,
        COUNT(*) AS num_clientes
    FROM gold.cliente_360
    WHERE tiene_actividad_minorista = TRUE
    GROUP BY canal_principal
    ORDER BY num_clientes DESC
""").fetchdf()
print(canales.to_string(index=False))

VALIDACIÓN 11 — Distribución por canal principal
canal_principal  num_clientes
      Temporada          2137
     Repetición           735
            B2B            67
       Muestras            57
     Devolución            32
  Dev. Muestras            12
            ECI             5
       Depósito             2
  RepetPdteProd             1
      Ecommerce             1
  Dev. Depósito             1
     PERSONALES             1


In [83]:
print("=" * 60)
print("ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 4)")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))
print(f"\nTotal columnas: {len(esquema)}")

ESQUEMA ACTUAL DE gold.cliente_360 (tras Bloque 4)
              column_name data_type
               id_cliente   VARCHAR
           nombre_cliente   VARCHAR
 nombre_comercial_cliente   VARCHAR
        localidad_cliente   VARCHAR
       codigo_postal_norm   VARCHAR
 codigo_provincia_cliente   VARCHAR
             tipo_mercado   VARCHAR
       es_cliente_espanol   BOOLEAN
                     pais   VARCHAR
             mosaic_grupo   VARCHAR
        mosaic_grupo_peso    DOUBLE
          mosaic_segmento   VARCHAR
     mosaic_segmento_peso    DOUBLE
              renta_media    DOUBLE
           perfil_premium   BOOLEAN
    perfil_familiar_joven   BOOLEAN
         perfil_turistico   BOOLEAN
             perfil_rural   BOOLEAN
   perfil_precio_sensible   BOOLEAN
             tiene_mosaic   BOOLEAN
          num_pedidos_b2b    BIGINT
          facturacion_b2b    DOUBLE
         ticket_medio_b2b    DOUBLE
         iva_aportado_b2b    DOUBLE
  importe_bruto_total_b2b    DOUBLE
 base_imponib

## 10. Bloque 5 — Temporal

Se enriquece la tabla con variables temporales calculadas a partir del cruce entre `silver.ventas_minoristas` y `silver.tiempo`. Estas variables capturan tres dimensiones complementarias del comportamiento del cliente:

**Recencia y antigüedad**: tiempo transcurrido desde la primera y última operación. La recencia es la variable clave para detectar churn (clientes inactivos) en los análisis posteriores.

**Frecuencia**: número medio de operaciones por mes activo, indicador de regularidad de compra.

**Estacionalidad**: distribución de operaciones entre las temporadas de moda (PV/OI), períodos de rebajas y eventos comerciales (Black Friday, San Valentín).

Todas las métricas se calculan excluyendo las devoluciones (`es_devolucion = FALSE`) para reflejar comportamiento de compra real, no movimientos de stock.

In [84]:
print("Añadiendo Bloque 5 (Temporal) a gold.cliente_360...\n")
print("⚠ Esto puede tardar 15-25 segundos (cruza ventas_minoristas con tiempo)\n")

con.execute("""
    CREATE OR REPLACE TABLE gold.cliente_360 AS
    WITH base AS (
        SELECT * FROM gold.cliente_360
    ),
    ventas_enriquecidas AS (
        -- Cruzamos ventas con el calendario
        SELECT
            v.id_cliente,
            v.fecha_venta,
            t.temporada_moda,
            t.es_rebajas,
            t.es_black_friday,
            t.es_san_valentin
        FROM silver.ventas_minoristas v
        INNER JOIN silver.tiempo t ON v.fecha_venta = t.fecha
        WHERE v.es_devolucion = FALSE
    ),
    metricas_temporales AS (
        SELECT
            id_cliente,
            
            -- Recencia y antigüedad
            MIN(fecha_venta)                                          AS fecha_primera_operacion,
            MAX(fecha_venta)                                          AS fecha_ultima_operacion,
            DATE_DIFF('day', MIN(fecha_venta), DATE '2025-12-31')     AS antiguedad_dias,
            DATE_DIFF('day', MAX(fecha_venta), DATE '2025-12-31')     AS recencia_dias,
            
            -- Frecuencia
            COUNT(DISTINCT DATE_TRUNC('month', fecha_venta))          AS meses_activo,
            
            -- Estacionalidad (temporadas de moda)
            ROUND(100.0 * SUM(CASE WHEN temporada_moda = 'PV' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_operaciones_pv,
            ROUND(100.0 * SUM(CASE WHEN temporada_moda = 'OI' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_operaciones_oi,
            
            -- Eventos comerciales
            ROUND(100.0 * SUM(CASE WHEN es_rebajas       THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_operaciones_rebajas,
            SUM(CASE WHEN es_black_friday  THEN 1 ELSE 0 END)         AS op_black_friday,
            SUM(CASE WHEN es_san_valentin  THEN 1 ELSE 0 END)         AS op_san_valentin
            
        FROM ventas_enriquecidas
        GROUP BY id_cliente
    )
    SELECT
        b.*,
        mt.fecha_primera_operacion,
        mt.fecha_ultima_operacion,
        mt.antiguedad_dias,
        mt.recencia_dias,
        COALESCE(mt.meses_activo, 0)              AS meses_activo,
        
        -- Frecuencia mensual: operaciones / meses activos
        CASE 
            WHEN COALESCE(mt.meses_activo, 0) = 0 THEN NULL
            ELSE ROUND(b.num_operaciones * 1.0 / mt.meses_activo, 2)
        END AS frecuencia_mensual,
        
        mt.pct_operaciones_pv,
        mt.pct_operaciones_oi,
        mt.pct_operaciones_rebajas,
        COALESCE(mt.op_black_friday, 0)            AS op_black_friday,
        COALESCE(mt.op_san_valentin, 0)            AS op_san_valentin
        
    FROM base b
    LEFT JOIN metricas_temporales mt ON b.id_cliente = mt.id_cliente
""")

print("✅ Bloque 5 añadido")
n = con.execute("SELECT COUNT(*) FROM gold.cliente_360").fetchone()[0]
n_cols = con.execute("""
    SELECT COUNT(*) FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
""").fetchone()[0]
print(f"   Filas: {n:,}")
print(f"   Columnas: {n_cols}  (esperado: 51)")

Añadiendo Bloque 5 (Temporal) a gold.cliente_360...

⚠ Esto puede tardar 15-25 segundos (cruza ventas_minoristas con tiempo)

✅ Bloque 5 añadido
   Filas: 3,469
   Columnas: 51  (esperado: 51)


## 11. Validación del Bloque 5

In [85]:
print("=" * 60)
print("VALIDACIÓN 12 — Estadísticas temporales globales")
print("=" * 60)

stats = con.execute("""
    SELECT 
        ROUND(AVG(antiguedad_dias), 0)         AS media_antiguedad_dias,
        ROUND(AVG(recencia_dias), 0)           AS media_recencia_dias,
        MIN(recencia_dias)                     AS min_recencia,
        MAX(recencia_dias)                     AS max_recencia,
        ROUND(AVG(meses_activo), 1)            AS media_meses_activo,
        ROUND(AVG(frecuencia_mensual), 1)      AS media_freq_mensual,
        ROUND(AVG(pct_operaciones_pv), 2)      AS media_pct_pv,
        ROUND(AVG(pct_operaciones_oi), 2)      AS media_pct_oi,
        ROUND(AVG(pct_operaciones_rebajas), 2) AS media_pct_rebajas
    FROM gold.cliente_360
    WHERE tiene_actividad_minorista = TRUE
""").fetchdf()
print(stats.T.to_string(header=False))

VALIDACIÓN 12 — Estadísticas temporales globales
media_antiguedad_dias  1178.00
media_recencia_dias     333.00
min_recencia              0.00
max_recencia           1458.00
media_meses_activo       14.30
media_freq_mensual       50.20
media_pct_pv             50.90
media_pct_oi             49.10
media_pct_rebajas        19.05


In [86]:
print("=" * 60)
print("VALIDACIÓN 13 — Segmentación por recencia (proxy de churn)")
print("=" * 60)

recencia = con.execute("""
    SELECT
        CASE
            WHEN recencia_dias IS NULL              THEN '0. Sin actividad'
            WHEN recencia_dias <= 90                THEN '1. Activos (≤90 días)'
            WHEN recencia_dias <= 180               THEN '2. En riesgo (91-180 días)'
            WHEN recencia_dias <= 365               THEN '3. Dormidos (181-365 días)'
            WHEN recencia_dias <= 730               THEN '4. Inactivos (1-2 años)'
            ELSE                                         '5. Perdidos (>2 años)'
        END AS segmento_recencia,
        COUNT(*) AS num_clientes,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gold.cliente_360
    GROUP BY segmento_recencia
    ORDER BY segmento_recencia
""").fetchdf()
print(recencia.to_string(index=False))

VALIDACIÓN 13 — Segmentación por recencia (proxy de churn)
         segmento_recencia  num_clientes   pct
          0. Sin actividad           439 12.65
     1. Activos (≤90 días)          1249 36.00
2. En riesgo (91-180 días)           555 16.00
3. Dormidos (181-365 días)           302  8.71
   4. Inactivos (1-2 años)           364 10.49
     5. Perdidos (>2 años)           560 16.14


In [87]:
print("=" * 60)
print("ESQUEMA FINAL DE gold.cliente_360")
print("=" * 60)
esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'gold' AND table_name = 'cliente_360'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))
print(f"\nTotal columnas: {len(esquema)}")

ESQUEMA FINAL DE gold.cliente_360
              column_name data_type
               id_cliente   VARCHAR
           nombre_cliente   VARCHAR
 nombre_comercial_cliente   VARCHAR
        localidad_cliente   VARCHAR
       codigo_postal_norm   VARCHAR
 codigo_provincia_cliente   VARCHAR
             tipo_mercado   VARCHAR
       es_cliente_espanol   BOOLEAN
                     pais   VARCHAR
             mosaic_grupo   VARCHAR
        mosaic_grupo_peso    DOUBLE
          mosaic_segmento   VARCHAR
     mosaic_segmento_peso    DOUBLE
              renta_media    DOUBLE
           perfil_premium   BOOLEAN
    perfil_familiar_joven   BOOLEAN
         perfil_turistico   BOOLEAN
             perfil_rural   BOOLEAN
   perfil_precio_sensible   BOOLEAN
             tiene_mosaic   BOOLEAN
          num_pedidos_b2b    BIGINT
          facturacion_b2b    DOUBLE
         ticket_medio_b2b    DOUBLE
         iva_aportado_b2b    DOUBLE
  importe_bruto_total_b2b    DOUBLE
 base_imponible_total_b2b    D

## 12. Validación cruzada final

Antes de cerrar la tabla, se realizan tres validaciones globales para garantizar la coherencia entre bloques y la integridad referencial.

In [88]:
print("=" * 70)
print("VALIDACIÓN 14 — Resumen de cobertura por bloque")
print("=" * 70)

cobertura = con.execute("""
    SELECT 
        'Total clientes'                                        AS bloque,
        COUNT(*)                                                AS num_clientes,
        ROUND(100.0 * COUNT(*) / 3469, 2)                       AS pct_cartera
    FROM gold.cliente_360
    UNION ALL
    SELECT 'Con MOSAIC (geomarketing)', 
           COUNT(*), 
           ROUND(100.0 * COUNT(*) / 3469, 2)
    FROM gold.cliente_360 WHERE tiene_mosaic = TRUE
    UNION ALL
    SELECT 'Con facturación B2B', 
           COUNT(*), 
           ROUND(100.0 * COUNT(*) / 3469, 2)
    FROM gold.cliente_360 WHERE tiene_facturacion_b2b = TRUE
    UNION ALL
    SELECT 'Con actividad minorista (comportamiento)', 
           COUNT(*), 
           ROUND(100.0 * COUNT(*) / 3469, 2)
    FROM gold.cliente_360 WHERE tiene_actividad_minorista = TRUE
    UNION ALL
    SELECT 'Cuentas corporativas (Ecommerce/ECI/Depósito)', 
           COUNT(*), 
           ROUND(100.0 * COUNT(*) / 3469, 2)
    FROM gold.cliente_360 WHERE es_cuenta_corporativa = TRUE
""").fetchdf()
print(cobertura.to_string(index=False))

VALIDACIÓN 14 — Resumen de cobertura por bloque
                                       bloque  num_clientes  pct_cartera
                               Total clientes          3469       100.00
                    Con MOSAIC (geomarketing)          2024        58.35
                          Con facturación B2B           234         6.75
     Con actividad minorista (comportamiento)          3051        87.95
Cuentas corporativas (Ecommerce/ECI/Depósito)            15         0.43


In [89]:
print("=" * 70)
print("VALIDACIÓN 15 — Top 10 clientes B2B y NACIONAL con cartera 360 completa")
print("=" * 70)

top_b2b = con.execute("""
    SELECT 
        id_cliente,
        nombre_cliente,
        pais,
        mosaic_grupo,
        num_pedidos_b2b,
        ROUND(facturacion_b2b, 0)  AS factur_b2b,
        unidades_vendidas,
        recencia_dias,
        ROUND(pct_pedidos_repeticion, 1) AS pct_repet
    FROM gold.cliente_360
    WHERE tiene_facturacion_b2b = TRUE 
      AND tipo_mercado = 'NACIONAL'
    ORDER BY facturacion_b2b DESC
    LIMIT 10
""").fetchdf()
print(top_b2b.to_string(index=False))

VALIDACIÓN 15 — Top 10 clientes B2B y NACIONAL con cartera 360 completa
id_cliente                         nombre_cliente   pais mosaic_grupo  num_pedidos_b2b  factur_b2b  unidades_vendidas  recencia_dias  pct_repet
      5728                  EL CORTE INGLES, S.A. España            A             1102   2285893.0           568085.0              1        6.6
      7571              BELLISIMA BEACHWEAR, S.L. España          NaN               50    757323.0           106163.0             49       21.8
      7409              SUEÑOS DELISTORES , S. L. España            G              608     97154.0            20334.0             16        6.8
     31738 VENTE PRIVEE.COM SA.SUCURSAL EN ESPAÑA España            E                2     86164.0            16163.0            396      100.0
     31165               WEB-VENTAS TIENDA ONLINE España            I               20     73785.0             7694.0            621       15.9
      5051         PRIVALIA VENTA DIRECTA, S.A.U. España        

## 13. Conclusiones del notebook 08

### Tabla analítica gold.cliente_360 construida

La tabla `gold.cliente_360` queda construida con **3.469 clientes** (uno por fila) y **51 variables** organizadas en cinco bloques temáticos:

| Bloque | Variables | Origen principal |
|---|---|---|
| 1. Identificación | 9 | silver.dim_cliente + silver.mapeo_paises |
| 2. Geográfico | 11 | silver.mosaic |
| 3. Económico (B2B) | 7 | silver.fact_lineas_pedido (con precio_unidad > 0) |
| 4. Comportamiento | 13 | silver.ventas_minoristas |
| 5. Temporal | 11 | silver.ventas_minoristas × silver.tiempo |

### Decisiones metodológicas aplicadas durante la construcción

**Bloque 2 — Geomarketing limitado a cartera nacional.** El JOIN con silver.mosaic se restringe a clientes con `tipo_mercado = 'NACIONAL'` para evitar coincidencias accidentales entre códigos postales españoles e internacionales con el mismo formato numérico. Cobertura efectiva: 2.024 clientes (58,3 % de la cartera total, 97,2 % de la nacional).

**Bloque 3 — Facturación restringida a pedidos B2B con precio negociado.** Tras detectar que `silver.fact_lineas_pedido` contiene 29.310 movimientos sin precio (muestras, depósitos, traspasos internos) frente a 4.104 pedidos con precio_unidad > 0, las métricas económicas se calculan exclusivamente sobre estos últimos. Esta decisión queda documentada en el informe de decisiones como caracterización del flujo B2B formal del ERP. La variable `tiene_facturacion_b2b` identifica los 234 clientes (6,7 % de la cartera) con esta facturación documentada.

**Bloque 4 — Métricas de comportamiento sobre operaciones reales.** Las variables de volumen, mix de canal y diversidad de SKU se calculan excluyendo devoluciones (`es_devolucion = FALSE`). Las devoluciones se contabilizan en variables separadas (num_operaciones_devol, unidades_devueltas, pct_ratio_devoluciones) para preservar la trazabilidad. Las 15 cuentas corporativas digitales (Ecommerce, Depósito, ECI) se identifican con el flag `es_cuenta_corporativa` para tratamiento diferenciado en los análisis posteriores.

**Bloque 5 — Recencia calculada respecto a 2025-12-31.** Se utiliza la fecha de cierre de la ventana de análisis como referencia para todas las métricas temporales. La segmentación por recencia define seis zonas (sin actividad, activos, en riesgo, dormidos, inactivos, perdidos) que servirán como variable base en el análisis de churn.

### Hallazgos clave para los análisis posteriores

**Estructura de la cartera por valor económico**: la concentración es muy alta. Los 234 clientes con facturación B2B documentada acumulan 7,77 millones de euros, de los cuales los 10 mayores aportan 6,7 millones (86 %). El Corte Inglés es el cliente individual con mayor facturación (2,3 M €).

**Patrón de fidelidad**: el cliente medio activo realiza un 31 % de operaciones de tipo Repetición (frente al 60 % de tipo Temporada), lo que sugiere un comportamiento de reposición consistente en aproximadamente un tercio de la actividad. Esta variable (`pct_pedidos_repeticion`) será central en el clustering y en el modelo de fidelidad.

**Estado de churn**: el 26,6 % de la cartera muestra señales de inactividad prolongada (más de un año desde la última operación), de los cuales el 16,1 % son clientes perdidos (más de dos años). Esto justifica plenamente el análisis de churn previsto en el TFG.

### Próximos pasos

Con gold.cliente_360 construida y validada, los notebooks siguientes (09 en adelante) pueden iniciar los análisis: descriptivo general, segmentación por clustering, modelización de fidelidad y geomarketing. Cada análisis filtrará la tabla según su población objetivo (cartera nacional, cartera con MOSAIC, clientes con facturación B2B, etc.) utilizando los flags binarios diseñados a tal efecto.

In [90]:
con.close()
print("=" * 60)
print("🎉 FASE GOLD COMPLETADA")
print("=" * 60)
print("\n✅ gold.cliente_360 construida con éxito")
print("   · 3.469 clientes")
print("   · 51 variables")
print("   · 5 bloques temáticos")
print("\n📋 Etapas completadas del proyecto:")
print("   ✅ Bronze (4 tablas cargadas)")
print("   ✅ Silver (5 tablas + 1 auxiliar: mapeo_paises)")
print("   ✅ Gold (cliente_360)")
print("\n🚀 SIGUIENTE FASE: análisis descriptivo + clustering + geomarketing + churn")

🎉 FASE GOLD COMPLETADA

✅ gold.cliente_360 construida con éxito
   · 3.469 clientes
   · 51 variables
   · 5 bloques temáticos

📋 Etapas completadas del proyecto:
   ✅ Bronze (4 tablas cargadas)
   ✅ Silver (5 tablas + 1 auxiliar: mapeo_paises)
   ✅ Gold (cliente_360)

🚀 SIGUIENTE FASE: análisis descriptivo + clustering + geomarketing + churn
